# 👑 Day 05: Constraint Search — N-Queens, Sudoku & Word Search

---

## The Constraint Satisfaction Pattern

Place items on a board under constraints. At each step: **Try → Validate → Recurse → Backtrack if invalid.**

The key to making these efficient: **prune early.** Don't wait until the end to discover a placement is invalid.

---

## 🎯 Problem 1: N-Queens (LC #51)

Place N queens on an N×N board so no two attack each other (same row, column, or diagonal).

**Approach:** Place one queen per row. For each row, try every column. Check constraints efficiently using sets.

**Three things to track:**
- Columns used
- Diagonals used (row - col is constant on each \ diagonal)
- Anti-diagonals used (row + col is constant on each / diagonal)

In [ ]:
def solveNQueens(n):
    """
    Backtracking with set-based constraint tracking.
    Time: O(N!), Space: O(N²)
    """
    result = []
    cols = set()
    diags = set()       # row - col
    anti_diags = set()  # row + col
    board = [['.' ] * n for _ in range(n)]
    
    def backtrack(row):
        if row == n:
            result.append(["".join(r) for r in board])
            return
        for col in range(n):
            if col in cols or (row - col) in diags or (row + col) in anti_diags:
                continue  # Conflict!
            # Place queen
            cols.add(col)
            diags.add(row - col)
            anti_diags.add(row + col)
            board[row][col] = 'Q'
            
            backtrack(row + 1)
            
            # Remove queen (backtrack)
            cols.remove(col)
            diags.remove(row - col)
            anti_diags.remove(row + col)
            board[row][col] = '.'
    
    backtrack(0)
    return result

solutions = solveNQueens(4)
for sol in solutions:
    for row in sol:
        print(row)
    print()

---

## 🎯 Problem 2: Sudoku Solver (LC #37)

Fill in the board so every row, column, and 3×3 box has digits 1-9.

**Same pattern:** Find empty cell → try 1-9 → validate → recurse → backtrack.

In [ ]:
def solveSudoku(board):
    """
    Constraint backtracking with set-based validation.
    """
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]  # box index = (r//3)*3 + c//3
    
    # Initialize from existing board
    for r in range(9):
        for c in range(9):
            if board[r][c] != '.':
                d = board[r][c]
                rows[r].add(d)
                cols[c].add(d)
                boxes[(r//3)*3 + c//3].add(d)
    
    def backtrack(pos):
        if pos == 81:
            return True  # Solved!
        r, c = pos // 9, pos % 9
        if board[r][c] != '.':
            return backtrack(pos + 1)  # Already filled
        
        box_idx = (r//3)*3 + c//3
        for d in '123456789':
            if d in rows[r] or d in cols[c] or d in boxes[box_idx]:
                continue
            # Place
            board[r][c] = d
            rows[r].add(d)
            cols[c].add(d)
            boxes[box_idx].add(d)
            
            if backtrack(pos + 1):
                return True
            
            # Backtrack
            board[r][c] = '.'
            rows[r].remove(d)
            cols[c].remove(d)
            boxes[box_idx].remove(d)
        
        return False  # No valid digit works here
    
    backtrack(0)

print("Sudoku solver ready — same pattern as N-Queens!")

---

## 🎯 Problem 3: Word Search (LC #79)

Given a grid and a word, can you find it by moving up/down/left/right to adjacent cells?

**DFS backtracking on grid.** Mark cell as visited, explore 4 directions, unmark.

In [ ]:
def exist(board, word):
    """
    Grid DFS backtracking.
    Time: O(m*n*4^L) where L = word length, Space: O(L)
    """
    m, n = len(board), len(board[0])
    
    def dfs(r, c, idx):
        if idx == len(word):
            return True
        if r < 0 or r >= m or c < 0 or c >= n or board[r][c] != word[idx]:
            return False
        
        # Mark visited (in-place trick: replace with #)
        temp = board[r][c]
        board[r][c] = '#'
        
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            if dfs(r + dr, c + dc, idx + 1):
                return True
        
        board[r][c] = temp  # Backtrack
        return False
    
    for r in range(m):
        for c in range(n):
            if dfs(r, c, 0):
                return True
    return False

board = [['A','B','C','E'],['S','F','C','S'],['A','D','E','E']]
print(exist(board, "ABCCED"))  # True
print(exist(board, "SEE"))     # True

---

## 🗺️ Constraint Search Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  CONSTRAINT SATISFACTION BACKTRACKING                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  TEMPLATE:                                                          ║
║    1. Find next empty position                                      ║
║    2. Try each possible value                                       ║
║    3. Check constraints (O(1) with sets!)                           ║
║    4. Place → recurse → backtrack if fails                         ║
║                                                                      ║
║  N-Queens: cols + diags (r-c) + anti-diags (r+c) sets             ║
║  Sudoku: rows[r] + cols[c] + boxes[idx] sets                      ║
║  Word Search: mark visited + 4-directional DFS                     ║
║                                                                      ║
║  KEY: O(1) constraint checking via precomputed sets                 ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 51 | N-Queens | Hard | Row-by-row + 3 sets |
| 37 | Sudoku Solver | Hard | Cell-by-cell + 3 sets |
| 79 | Word Search | Medium | Grid DFS + in-place visited |
| 212 | Word Search II | Hard | Trie + grid backtracking |
| 52 | N-Queens II | Hard | Same as 51, just count |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Constraint search = try → validate → recurse → backtrack      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Use SETS for O(1) constraint checking                          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Grid backtracking: mark cell visited, explore, unmark          │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Backtracking & Pruning